A notebook to compute average "partisan bias" scores by state & chamber

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from knobs_functions import *
import warnings

warnings.filterwarnings('ignore')

Calculate the average value of "partisan bias" metrics by state & chamber
Note: Switch the list of ensembles for just the A0 table

In [2]:
# NOTE - This does not generate the "all variants together" table right now. Already have the A0 table.

from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)

ensembles = ["base0", "pop_minus", "pop_plus", "distpair", "ust", "distpair_ust", "reversible", "county25", "county50", "county75", "county100"]
table_list = [[x] for x in ensembles if x not in ["base0"]]
tables: Dict[str, Dict[Tuple[str, str], Any]] = dict()

for variants in table_list:
    bias_table: Dict[Tuple[str, str], Any] = dict()

    for state, chamber in state_chamber_list:
        bias_table[(state, chamber)] = dict()
        for m in metrics:
            all_values: List[float] = []
            for e in variants:
                arr = fetch_score_array(state, chamber, e, m)
                all_values.extend(arr)
            # Guard for undefined declinations
            all_values = [x for x in all_values if not np.isnan(x)]
            mean_value = np.mean(all_values)
            bias_table[(state, chamber)][m] = mean_value

    tables[variants[0]] = bias_table


Convert the dict to a pandas DataFrame and LaTex

TODO's
* Need to generate the LaTex columns as right-justified
* Except the header's which should be centered

I tweaked both by hand

In [3]:
def make_partisan_bias_table(bias_table: Dict[Tuple[str, str], Any], *, 
                             latex_filename=None, 
                             markdown_filename=None,
                             rounding: int = 2):

    index_list = [f'{a[0]} {a[1]}' for a in state_chamber_list]
    df = pd.DataFrame(columns=metrics, index=index_list)

    for state, chamber in state_chamber_list:
        for m in metrics:
            multiplier = 1 if m == "declination" else 100
            df.loc[f'{state} {chamber}', m] = bias_table[(state, chamber)][m] * multiplier
    df = df.applymap(pd.to_numeric)
    df = df.round(rounding)
    
    # Prepare state/chamber labels with seat counts
    state_chamber_size_dict = {f'{state} {chamber}': f'{state} {num_seats_dict[(state, chamber)]}' 
                              for state, chamber in state_chamber_list}
    
    # Greek letters for different formats
    greek_latex = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }
    
    greek_unicode = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }
    
    # Metric name mappings
    metrics_name_dict_latex = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": f"${greek_latex['beta']}$",
        "seats_bias": f"${greek_latex['alpha']}_s$",
        "votes_bias": f"${greek_latex['alpha']}_v$",
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek_latex['delta']
    }
    
    metrics_name_dict_markdown = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": greek_unicode['beta'],
        "seats_bias": greek_unicode['alpha'] + "ₛ",  # Unicode subscript
        "votes_bias": greek_unicode['alpha'] + "ᵥ",  # Unicode subscript
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek_unicode['delta']
    }
    
    # Generate LaTeX table
    if latex_filename is not None:
        df_latex = df.copy()
        df_latex = df_latex.applymap(lambda x: f"{x:.2f}")
        
        # Add LaTeX color formatting
        for state, chamber in state_chamber_list:
            for m in metrics:
                val = df.loc[f'{state} {chamber}', m]
                df_latex.loc[f'{state} {chamber}', m] = f'\\textcolor{{black}}{{ {val:.2f} }}'
        
        df_latex.rename(columns=metrics_name_dict_latex, index=state_chamber_size_dict, inplace=True)
        df_latex.to_latex(latex_filename, escape=False)
    
    # Generate Markdown table
    if markdown_filename is not None:
        df_markdown = df.copy()
        df_markdown = df_markdown.applymap(lambda x: f"{x:.2f}")
        df_markdown.rename(columns=metrics_name_dict_markdown, index=state_chamber_size_dict, inplace=True)
        
        # Create markdown table manually for better control
        with open(markdown_filename, 'w', encoding='utf-8') as f:
            # Write header
            headers = ['District'] + list(df_markdown.columns)
            f.write('| ' + ' | '.join(headers) + ' |\n')
            f.write('|' + '|'.join(['-' * (len(h) + 2) for h in headers]) + '|\n')
            
            # Write data rows
            for idx in df_markdown.index:
                row = [idx] + [str(df_markdown.loc[idx, col]) for col in df_markdown.columns]
                f.write('| ' + ' | '.join(row) + ' |\n')

    return df

Note: Switch the output location for the LaTeX for the A0 table

In [4]:
for variant in tables:
    bias_table = tables[variant]
    latex_out: str = f'temp/partisan_bias_table_{variant}.tex'
    markdown_out: str = f'temp/partisan_bias_table_{variant}.md'
    make_partisan_bias_table(bias_table, latex_filename=latex_out, markdown_filename=markdown_out)